# 격자 GeoJSON + 종합위험지수 조인

- 입력 1: `data/01._격자_(4개_시·구).geojson` — 전체 격자 폴리곤 (gid)
- 입력 2: `output/08_격자_종합위험지수.csv` — 격자별 종합위험지수 (grid_gid)
- 조인 키: `gid` == `grid_gid`
- 조인 컬럼: `entropy_composite_risk`
- 출력: `output/grid_entropy_risk.geojson`

In [ ]:
import os
import geopandas as gpd
import pandas as pd

BASE_DIR   = os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

GRID_FILE = os.path.join(DATA_DIR, "01._격자_(4개_시·구).geojson")
RISK_FILE = os.path.join(OUTPUT_DIR, "08_격자_종합위험지수.csv")
OUT_FILE  = os.path.join(OUTPUT_DIR, "grid_entropy_risk.geojson")

print(f"BASE_DIR  : {BASE_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# 1. 격자 GeoJSON 로드
print("[1] 격자 GeoJSON 로드...")
gdf = gpd.read_file(GRID_FILE)
print(f"    격자 수: {len(gdf):,}개")
print(f"    컬럼: {list(gdf.columns)}")
print(f"    CRS: {gdf.crs}")
gdf.head(3)

In [ ]:
# 2. 종합위험지수 CSV 로드
print("[2] 종합위험지수 CSV 로드...")
df_risk = pd.read_csv(RISK_FILE, encoding="utf-8-sig")
print(f"    행: {len(df_risk):,}개")
print(f"    컬럼: {list(df_risk.columns)}")
df_risk[["grid_gid", "entropy_composite_risk", "entropy_rank"]].head(3)

In [ ]:
# 3. LEFT JOIN: 격자(gid) ← 위험지수(grid_gid)
print("[3] JOIN 수행...")
gdf_joined = gdf.merge(
    df_risk[["grid_gid", "entropy_composite_risk", "entropy_rank"]],
    left_on="gid",
    right_on="grid_gid",
    how="left"
).drop(columns=["grid_gid"])

matched = gdf_joined["entropy_composite_risk"].notna().sum()
print(f"    전체 격자: {len(gdf_joined):,}개")
print(f"    위험지수 매칭: {matched:,}개 ({matched/len(gdf_joined)*100:.1f}%)")
print(f"    미매칭(NaN): {len(gdf_joined)-matched:,}개")
gdf_joined.head(3)

In [ ]:
# 4. 결과 저장
print("[4] GeoJSON 저장...")
gdf_joined.to_file(OUT_FILE, driver="GeoJSON")
print(f"    저장 완료: {OUT_FILE}")
print(f"    컬럼: {list(gdf_joined.columns)}")

In [ ]:
# 5. 결과 확인
print("[5] entropy_composite_risk 분포")
print(gdf_joined["entropy_composite_risk"].describe())